# Leveraged HRP — Risk Parity with Cash Borrowing
**QFGB 8948 — Team Portfolio Project · Extension: Leveraged Risk Parity**

Unlevered HRP passes the client's −15% drawdown constraint comfortably (max DD −3.79%)  
but falls short of the 6.02% required return at 4.55%. This notebook explores whether  
applying leverage — financed by shorting cash at FRED Fed Funds rates — can close that gap.

| Strategy | Mechanism |
|---|---|
| **HRP 1x** | Baseline unlevered HRP |
| **HRP 1.5x Fixed** | Borrow 50% at Fed Funds + 25bp; invest 150% in HRP |
| **HRP 2.0x Fixed** | Borrow 100% at Fed Funds + 25bp; invest 200% in HRP |
| **HRP VolTgt 8%** | Dynamic leverage targeting 8% ann. vol (capped at 3x) |
| **HRP VolTgt 10%** | Dynamic leverage targeting 10% ann. vol (capped at 3x) |

**Net monthly return formula:**
$$r_{\text{net},t} = L_t \cdot r_{\text{HRP},t} - (L_t - 1) \cdot r_{\text{borrow},t}$$

where $r_{\text{borrow}}$ = monthly Fed Funds rate + 25bp annual spread.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from scipy.optimize import minimize
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform
from IPython.display import display
import pandas_datareader.data as web

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})
np.random.seed(42)

ANN      = 12
ROLL_WIN = 36
MAX_LEV  = 3.0   # leverage cap for vol-targeted strategies
SPREAD   = 0.0025  # 25bp annual spread over Fed Funds for borrowing cost

# Client goals
REQUIRED_RETURN  = 0.0602
MAX_DRAWDOWN     = -0.15

COLORS = {
    'HRP 1x':        '#888888',
    'HRP 1.5x':      '#2E75B6',
    'HRP 2.0x':      '#ED7D31',
    'VolTgt 8%':     '#70AD47',
    'VolTgt 10%':    '#9B59B6',
    'Equal Weight':  '#C00000',
}

---
## 1 · Data Loading

In [ ]:
EXCEL_PATH = '1 Project - Data.xlsx'

# Asset returns
raw = pd.read_excel(EXCEL_PATH, sheet_name='FULL DATA', header=None)
r_all = raw.iloc[2:, 0:6].copy()
r_all.columns = ['Date','Asset 1','Asset 2','Asset 3','Asset 4','Asset 5']
r_all = r_all.dropna(subset=['Date']).set_index('Date')
for c in r_all.columns:
    r_all[c] = pd.to_numeric(r_all[c], errors='coerce')
r_all = r_all.dropna()
r_all.index = pd.date_range(start='2004-01-31', periods=len(r_all), freq='ME')
T, N = r_all.shape
assets = r_all.columns.tolist()
EQ_W   = np.ones(N) / N

# Factor sheet (for Cash reference)
raw_f = pd.read_excel(EXCEL_PATH, sheet_name='FACTORS', header=None)
col_names = ['Date'] + raw_f.iloc[0, 1:].tolist()
F_all = raw_f.iloc[3:, :len(col_names)].copy()
F_all.columns = col_names
F_all = F_all.set_index('Date')
for c in F_all.columns:
    F_all[c] = pd.to_numeric(F_all[c], errors='coerce')
F_all = F_all.dropna(how='all')
F_all.index = pd.date_range(start='2004-01-31', periods=len(F_all), freq='ME')

print(f'Assets : {assets}')
print(f'Range  : {r_all.index[0].strftime("%b %Y")} → {r_all.index[-1].strftime("%b %Y")}  ({T} months)')

---
## 2 · FRED Borrowing Rates

Fetch the **daily Effective Federal Funds Rate (DFF)** and **3-Month T-Bill (DTB3)**  
from FRED via `pandas_datareader`. Resample to month-end, convert annualised % → monthly decimal,  
then add a 25bp annual spread to proxy realistic institutional short-term borrowing cost.

In [ ]:
print('Fetching FRED rates...')
fred_daily = web.DataReader(['DFF', 'DTB3'], 'fred', '2003-06-01', '2014-02-28')
fred_daily.columns = ['Fed Funds (ann %)', '3M T-Bill (ann %)']

# Month-end values, forward-fill weekends/holidays
fred_monthly = fred_daily.resample('ME').last().ffill()

# Annualised % → monthly decimal  (e.g. 5.00% annual → 5/100/12 per month)
fred_m = fred_monthly / 100 / 12
fred_m.index = fred_m.index.to_period('M').to_timestamp('M')
fred_m = fred_m.reindex(r_all.index, method='ffill')

# Borrowing rate = Fed Funds + 25bp spread (monthly)
r_borrow = fred_m['Fed Funds (ann %)'] + SPREAD / 12

# Summary by period
periods = {
    'Full dataset (2004-2013)': slice('2004', '2013'),
    'Pre-GFC    (2004-2006)':   slice('2004', '2006'),
    'GFC        (2007-2009)':   slice('2007', '2009'),
    'Post-GFC   (2010-2013)':   slice('2010', '2013'),
}
rows = []
for label, sl in periods.items():
    ff  = fred_m.loc[sl, 'Fed Funds (ann %)'].mean() * 12
    tb  = fred_m.loc[sl, '3M T-Bill (ann %)'].mean() * 12
    br  = r_borrow.loc[sl].mean() * 12
    rows.append({'Period': label, 'Fed Funds p.a.': ff, '3M T-Bill p.a.': tb, 'Borrow rate p.a.': br})

rates_df = pd.DataFrame(rows).set_index('Period')
display(rates_df.style.format('{:.2%}').set_caption('FRED Rates (annualised)'))

# Plot Fed Funds over time
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(fred_m.index, fred_m['Fed Funds (ann %)'] * 12 * 100,
        color='steelblue', lw=1.5, label='Fed Funds Rate (ann. %)')
ax.plot(fred_m.index, fred_m['3M T-Bill (ann %)'] * 12 * 100,
        color='tomato', lw=1.2, ls='--', label='3M T-Bill (ann. %)')
ax.axvspan(pd.Timestamp('2007-12-01'), pd.Timestamp('2009-06-30'),
           color='lightcoral', alpha=0.2, label='GFC')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator(1))
ax.set_ylabel('Rate (%)')
ax.set_title('FRED Borrowing Rates (2004–2013)', fontweight='bold')
ax.legend(fontsize=9); sns.despine(ax=ax)
plt.tight_layout()
plt.savefig('fred_rates.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3 · Rolling HRP Weights & Portfolio Volatility

In [ ]:
def hrp_portfolio(cov):
    """Hierarchical Risk Parity via single-linkage clustering + recursive bisection."""
    n    = cov.shape[0]
    d    = np.diag(1.0 / np.sqrt(np.diag(cov)))
    corr = d @ cov @ d
    np.fill_diagonal(corr, 1.0)
    dist = np.sqrt(np.clip(0.5 * (1 - corr), 0, None))
    dist = (dist + dist.T) / 2
    np.fill_diagonal(dist, 0)
    Z     = linkage(squareform(dist), method='single')
    order = leaves_list(Z).tolist()
    weights = np.ones(n)

    def bisect(items, wts):
        if len(items) <= 1:
            return
        mid   = len(items) // 2
        L, R  = items[:mid], items[mid:]
        def cv(idx):
            sub = cov[np.ix_(idx, idx)]
            iv  = 1.0 / np.diag(sub)
            w   = iv / iv.sum()
            return float(w @ sub @ w)
        vl, vr = cv(L), cv(R)
        alpha  = 1.0 - vl / (vl + vr)
        wts[L] *= alpha
        wts[R] *= (1.0 - alpha)
        bisect(L, wts)
        bisect(R, wts)

    bisect(order, weights)
    return weights / weights.sum()


# Rolling 36-month HRP weights + portfolio vol estimate
hrp_w      = np.full((T, N), np.nan)
rolling_vol = np.full(T, np.nan)

for t in range(ROLL_WIN - 1, T):
    cov_w          = np.cov(r_all.iloc[t - ROLL_WIN + 1 : t + 1].values.T, ddof=0)
    hrp_w[t]       = hrp_portfolio(cov_w)
    w              = hrp_w[t]
    rolling_vol[t] = np.sqrt(w @ cov_w @ w) * np.sqrt(ANN)   # annualised

hrp_wdf     = pd.DataFrame(hrp_w,      index=r_all.index, columns=assets)
rv_series   = pd.Series(rolling_vol,   index=r_all.index, name='Rolling Vol (ann.)')

# Unlevered HRP returns (weights at t applied to returns at t+1)
wl        = hrp_wdf.shift(1)
valid     = wl.notna().all(axis=1)
r_hrp     = (wl * r_all).sum(axis=1)[valid]
bt_start  = r_hrp.index[0]

print(f'Backtest window : {bt_start.strftime("%b %Y")} → {r_all.index[-1].strftime("%b %Y")}  ({len(r_hrp)} months)')
print(f'Avg rolling vol : {rv_series.dropna().mean():.2%}  (annualised)')
print(f'Min rolling vol : {rv_series.dropna().min():.2%}')
print(f'Max rolling vol : {rv_series.dropna().max():.2%}')

# Plot rolling vol
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(rv_series.dropna().index, rv_series.dropna() * 100,
        color='steelblue', lw=1.5)
ax.axhline(rv_series.dropna().mean() * 100, color='tomato', ls='--', lw=1,
           label=f'Mean {rv_series.dropna().mean():.2%}')
ax.axvspan(pd.Timestamp('2007-12-01'), pd.Timestamp('2009-06-30'),
           color='lightcoral', alpha=0.2, label='GFC')
ax.set_title('HRP Rolling 36-Month Annualised Portfolio Volatility', fontweight='bold')
ax.set_ylabel('Vol (%)')
ax.legend(fontsize=9); sns.despine(ax=ax)
plt.tight_layout()
plt.savefig('hrp_rolling_vol.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4 · Leverage Strategies

**Fixed leverage:** multiply every month's return by a constant $L$, finance the excess at the borrowing rate.

**Vol-targeting:** set $L_t = \min\left(\dfrac{\sigma^*}{\hat{\sigma}_{t-1}},\ L_{\max}\right)$  
where $\hat{\sigma}_{t-1}$ is the prior month's rolling 36-month annualised vol estimate.  
This keeps realised portfolio vol close to the target $\sigma^*$, scaling down in volatile periods.

In [ ]:
def levered_returns(r_base, leverage_series, r_borrow_series):
    """r_net = L * r_base - (L - 1) * r_borrow"""
    lev = leverage_series.reindex(r_base.index).fillna(1.0)
    rb  = r_borrow_series.reindex(r_base.index).fillna(0.0)
    return lev * r_base - (lev - 1) * rb


rv_lag = rv_series.shift(1)   # prior month vol — no lookahead

lev_15   = pd.Series(1.5,  index=r_hrp.index)
lev_20   = pd.Series(2.0,  index=r_hrp.index)
lev_vt8  = (0.08  / rv_lag).clip(1.0, MAX_LEV).reindex(r_hrp.index).fillna(1.0)
lev_vt10 = (0.10  / rv_lag).clip(1.0, MAX_LEV).reindex(r_hrp.index).fillna(1.0)

strategies = {
    'HRP 1x':      (r_hrp,                                                pd.Series(1.0, index=r_hrp.index)),
    'HRP 1.5x':    (levered_returns(r_hrp, lev_15,   r_borrow),           lev_15),
    'HRP 2.0x':    (levered_returns(r_hrp, lev_20,   r_borrow),           lev_20),
    'VolTgt 8%':   (levered_returns(r_hrp, lev_vt8,  r_borrow),           lev_vt8),
    'VolTgt 10%':  (levered_returns(r_hrp, lev_vt10, r_borrow),           lev_vt10),
    'Equal Weight':(pd.DataFrame(np.tile(EQ_W, (T,1)), index=r_all.index, columns=assets)
                      .shift(1).mul(r_all).sum(axis=1)[valid],             pd.Series(1.0, index=r_hrp.index)),
}

# Leverage stats for vol-targeted
print('Vol-Targeted Leverage Statistics (backtest period):')
for name in ['VolTgt 8%', 'VolTgt 10%']:
    lev = strategies[name][1].loc[bt_start:]
    print(f'  {name}: min={lev.min():.2f}x  avg={lev.mean():.2f}x  max={lev.max():.2f}x  '
          f'time-at-cap={( lev >= MAX_LEV).mean():.1%}')

---
## 5 · Performance Analysis

In [ ]:
rf_bt = r_borrow.reindex(r_hrp.index).fillna(0)

def perf_metrics(rets, rf_series):
    rf     = rf_series.reindex(rets.index).mean() * ANN
    mu     = rets.mean() * ANN
    vol    = rets.std()  * np.sqrt(ANN)
    sharpe = (mu - rf) / vol
    cum    = (1 + rets).cumprod()
    maxdd  = (cum / cum.expanding().max() - 1).min()
    calmar = mu / abs(maxdd) if maxdd < 0 else float('nan')
    return {'Ann. Return': mu, 'Ann. Vol': vol, 'Sharpe': sharpe,
            'Max DD': maxdd, 'Calmar': calmar,
            'Worst Month': rets.min(), 'Best Month': rets.max()}


rows = []
for name, (rets, lev_s) in strategies.items():
    r_bt  = rets.loc[bt_start:]
    pm    = perf_metrics(r_bt, rf_bt)
    pm['Avg Leverage'] = float(lev_s.loc[bt_start:].mean())
    pm['Pass DD?']     = 'YES ✓' if pm['Max DD'] > MAX_DRAWDOWN   else 'NO ✗'
    pm['Meet Return?'] = 'YES ✓' if pm['Ann. Return'] >= REQUIRED_RETURN else 'NO ✗'
    pm['Strategy']     = name
    rows.append(pm)

perf_df = pd.DataFrame(rows).set_index('Strategy')
fmt = {
    'Ann. Return': '{:.2%}', 'Ann. Vol': '{:.2%}', 'Sharpe': '{:.3f}',
    'Max DD': '{:.2%}',      'Calmar': '{:.3f}',
    'Worst Month': '{:.2%}', 'Best Month': '{:.2%}', 'Avg Leverage': '{:.2f}x',
}

def highlight(row):
    styles = [''] * len(row)
    idx = list(row.index)
    if row.get('Ann. Return', 0) >= REQUIRED_RETURN:
        styles[idx.index('Ann. Return')] = 'background-color: #E2EFDA'
    else:
        styles[idx.index('Ann. Return')] = 'background-color: #FFDDD9'
    if row.get('Max DD', 0) > MAX_DRAWDOWN:
        styles[idx.index('Max DD')] = 'background-color: #E2EFDA'
    else:
        styles[idx.index('Max DD')] = 'background-color: #FFDDD9'
    return styles

display(
    perf_df.drop(columns=['Pass DD?','Meet Return?'])
           .style.apply(highlight, axis=1)
           .format(fmt)
           .set_caption(
               f'Backtest Performance ({bt_start.strftime("%b %Y")} – Dec 2013) | '
               f'Borrow: Fed Funds + 25bp | Max Leverage: {MAX_LEV:.0f}x | '
               f'Green = meets client goal'
           )
)

---
## 6 · Cumulative Returns & Drawdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ls_map = {'Equal Weight': '--', 'HRP 1x': ':'}

for name, (rets, _) in strategies.items():
    r_bt = rets.loc[bt_start:]
    cum  = 100 * (1 + r_bt).cumprod()
    dd   = (1 + r_bt).cumprod() / (1 + r_bt).cumprod().expanding().max() - 1
    ls   = ls_map.get(name, '-')
    lw   = 1.2 if name in ('Equal Weight', 'HRP 1x') else 1.8
    col  = COLORS.get(name, 'gray')
    axes[0].plot(cum.index, cum.values,  label=name, color=col, lw=lw, ls=ls)
    axes[1].plot(dd.index,  dd.values*100, label=name, color=col, lw=lw, ls=ls)

axes[1].axhline(-15, color='red', lw=1.2, ls=':', alpha=0.8, label='−15% client limit')
axes[1].axvspan(pd.Timestamp('2007-12-01'), pd.Timestamp('2009-06-30'),
                color='lightcoral', alpha=0.15)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
    ax.tick_params(axis='x', labelrotation=45, labelsize=8)
    ax.legend(fontsize=8)
    sns.despine(ax=ax)

axes[0].set_title('Cumulative Return (Jan 2007 = 100)', fontweight='bold')
axes[0].set_ylabel('Indexed Return')
axes[1].set_title('Drawdown (%)', fontweight='bold')
axes[1].set_ylabel('Drawdown (%)')
fig.suptitle('Leveraged HRP — Backtest Jan 2007 – Dec 2013', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('lev_hrp_backtest.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7 · Year-by-Year Returns

In [ ]:
yearly_rows = []
for yr in range(2007, 2014):
    row = {'Year': str(yr)}
    for name, (rets, _) in strategies.items():
        sub = rets[rets.index.year == yr]
        row[name] = float((1 + sub).prod() - 1) if len(sub) > 0 else float('nan')
    row['Fed Funds (avg p.a.)'] = float(
        fred_m.loc[fred_m.index.year == yr, 'Fed Funds (ann %)'].mean() * 12
    )
    yearly_rows.append(row)

yearly_df = pd.DataFrame(yearly_rows).set_index('Year')

def yr_color(val):
    if pd.isna(val): return ''
    return 'background-color: #E2EFDA' if val > 0 else 'background-color: #FFDDD9'

fmt_yr = {c: '{:.2%}' for c in yearly_df.columns}
display(
    yearly_df.style
             .applymap(yr_color, subset=[c for c in yearly_df.columns if c != 'Fed Funds (avg p.a.)'])
             .format(fmt_yr)
             .set_caption('Annual Returns by Strategy  |  Green = positive  |  Red = negative')
)

---
## 8 · Dynamic Leverage Over Time (Vol-Targeted Strategies)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

# Top: rolling vol
rv_bt = rv_series.loc[bt_start:]
axes[0].plot(rv_bt.index, rv_bt * 100, color='steelblue', lw=1.5)
axes[0].axhline(rv_bt.mean() * 100, color='steelblue', ls='--', lw=0.8, alpha=0.6)
axes[0].axvspan(pd.Timestamp('2007-12-01'), pd.Timestamp('2009-06-30'),
                color='lightcoral', alpha=0.15, label='GFC')
axes[0].set_title('Rolling 36-Month HRP Portfolio Volatility (Annualised)', fontweight='bold')
axes[0].set_ylabel('Vol (%)')
axes[0].legend(fontsize=9)

# Bottom: leverage levels
for name, col, tgt in [('VolTgt 8%', '#70AD47', '8%'), ('VolTgt 10%', '#9B59B6', '10%')]:
    lev = strategies[name][1].loc[bt_start:]
    axes[1].plot(lev.index, lev.values, color=col, lw=1.5, label=f'VolTgt {tgt}')

axes[1].axhline(1.5, color='#2E75B6', ls='--', lw=1.0, alpha=0.7, label='1.5x fixed')
axes[1].axhline(2.0, color='#ED7D31', ls='--', lw=1.0, alpha=0.7, label='2.0x fixed')
axes[1].axhline(MAX_LEV, color='red', ls=':', lw=1.0, alpha=0.6, label=f'{MAX_LEV:.0f}x cap')
axes[1].set_title('Leverage Applied Each Month', fontweight='bold')
axes[1].set_ylabel('Leverage (x)')
axes[1].legend(fontsize=9, ncol=2)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b-%y'))
axes[1].xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
axes[1].tick_params(axis='x', labelrotation=45, labelsize=8)

for ax in axes:
    sns.despine(ax=ax)

plt.tight_layout()
plt.savefig('lev_hrp_leverage.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9 · Key Findings & Caveats

### Findings

| Strategy | Meets Return (≥6.02%)? | Passes DD (≥−15%)? |
|---|---|---|
| HRP 1x (Unlevered) | ❌ 4.55% | ✅ −3.79% |
| **HRP 1.5x Fixed** | **✅ 6.17%** | **✅ −5.70%** |
| HRP 2.0x Fixed | ✅ 7.78% | ✅ −7.59% |
| VolTgt 8% | ✅ 10.56% | ✅ −11.30% |
| VolTgt 10% | ✅ 11.02% | ✅ −11.30% |
| Equal Weight | ❌ 4.25% | ❌ −23.66% |

**HRP 1.5x is the minimum leverage that satisfies both client goals** while maintaining substantial  
buffer from the −15% drawdown limit (−5.70% max DD, 9.3% of buffer remaining).

### Caveats

1. **Artificially cheap borrowing.** Fed Funds averaged just **1.06% p.a.** over 2007–2013 due to  
   post-GFC ZIRP. At pre-2007 levels (~5%), leverage is significantly more expensive.

2. **Bond tailwind.** HRP is heavily bond-weighted. 2004–2013 was a strong era for fixed income  
   (broadly falling rates). Leverage amplified a regime-specific tailwind.

3. **Vol-targeted strategies cap at 3x.** HRP's vol is so low (~2.87% ann.) that hitting a  
   10% vol target requires ~3.5x leverage — beyond the cap. VolTgt 10% runs at 3x constantly.

4. **2013 Taper Tantrum.** All levered strategies lost money in 2013 when the Fed signalled  
   tapering QE, causing bonds to sell off sharply. Losses were small (−3.22% for 1.5x) but  
   illustrate the sensitivity of the bond-heavy HRP to rate normalisation.

5. **Client suitability.** Leverage is not automatically appropriate for a 57-year-old approaching  
   retirement. This analysis is exploratory — the core recommendation remains unlevered HRP or MVO.